In [ ]:
# Cell 1: Chạy LLM Judge cho Dense retrieval trên 100 query bằng Qwen API
import json
import logging
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "backend":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.rag.evaluation.llm_judge_retrieval import LLMJudgeConfig, configure_console_encoding, iter_jsonl, load_dotenv
from backend.rag.evaluation.run_llm_judge_three_retrievers import judge_single_retriever

def configure_qwen_api():
    """Cấu hình Qwen API qua OpenRouter hoặc endpoint OpenAI-compatible khác."""
    load_dotenv()
    if os.environ.get("OPENROUTER_API_KEY") and not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]
    os.environ.pop("GITHUB_MODELS_ENDPOINT", None)
    os.environ.pop("GITHUB_TOKEN", None)
    os.environ.setdefault("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
    os.environ.setdefault("CHAT_MODEL", "qwen/qwen3-14b")
    if not os.environ.get("OPENAI_API_KEY"):
        raise ValueError("Thiếu OPENAI_API_KEY hoặc OPENROUTER_API_KEY trong .env để gọi Qwen API.")

configure_console_encoding()
configure_qwen_api()
logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(message)s")

OUTPUT_DIR = Path("report/evaluate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QUERIES = iter_jsonl(Path("data/evaluation/traveler_need_queries_500.jsonl"))[:100]
BASE_CONFIG = LLMJudgeConfig(
    queries_path=Path("data/evaluation/traveler_need_queries_500.jsonl"),
    output_path=OUTPUT_DIR / "unused.jsonl",
    summary_path=OUTPUT_DIR / "unused_summary.json",
    top_k=5,
    candidate_k=30,
    max_context_chars=5000,
    limit=100,
    dry_run=False,
    device="cpu",
)

dense_payload = judge_single_retriever(
    retriever_name="dense",
    queries=QUERIES,
    base_config=BASE_CONFIG,
    output_dir=OUTPUT_DIR,
    resume=True,
    sleep_seconds=0.2,
    output_label="100_qwen_api",
)
print(json.dumps({"file": "report/evaluate/llm_judge_dense_100_qwen_api.json", "summary": dense_payload["summary"]}, ensure_ascii=False, indent=2))

In [ ]:
# Cell 2: Chạy LLM Judge cho BM25 retrieval trên 100 query bằng Qwen API
bm25_payload = judge_single_retriever(
    retriever_name="bm25",
    queries=QUERIES,
    base_config=BASE_CONFIG,
    output_dir=OUTPUT_DIR,
    resume=True,
    sleep_seconds=0.2,
    output_label="100_qwen_api",
)
print(json.dumps({"file": "report/evaluate/llm_judge_bm25_100_qwen_api.json", "summary": bm25_payload["summary"]}, ensure_ascii=False, indent=2))

In [ ]:
# Cell 3: Chạy LLM Judge cho Hybrid retrieval trên 100 query và tạo file tổng hợp scores-only
from backend.rag.evaluation.run_llm_judge_three_retrievers import write_combined_scores

hybrid_payload = judge_single_retriever(
    retriever_name="hybrid",
    queries=QUERIES,
    base_config=BASE_CONFIG,
    output_dir=OUTPUT_DIR,
    resume=True,
    sleep_seconds=0.2,
    output_label="100_qwen_api",
)

dense_payload = json.loads(Path("report/evaluate/llm_judge_dense_100_qwen_api.json").read_text(encoding="utf-8"))
bm25_payload = json.loads(Path("report/evaluate/llm_judge_bm25_100_qwen_api.json").read_text(encoding="utf-8"))
retriever_payloads = {"dense": dense_payload, "bm25": bm25_payload, "hybrid": hybrid_payload}
write_combined_scores(
    queries=QUERIES,
    retriever_payloads=retriever_payloads,
    output_path=Path("report/evaluate/llm_judge_scores_only_100_qwen_api.json"),
)
print(json.dumps({
    "hybrid_file": "report/evaluate/llm_judge_hybrid_100_qwen_api.json",
    "combined_scores_file": "report/evaluate/llm_judge_scores_only_100_qwen_api.json",
    "summary": hybrid_payload["summary"],
}, ensure_ascii=False, indent=2))